In [1]:
!pip install pandas numpy scikit-learn xgboost matplotlib seaborn

In [7]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import zipfile

In [8]:
with zipfile.ZipFile("/content/train.csv.zip", "r") as zip_ref:
    zip_ref.extractall("/content/")

train = pd.read_csv('/content/train.csv')
store = pd.read_csv('/content/store.csv')

print("Treino:")
print(train.head())

print("\nLojas:")
print(store.head())

Treino:
   Store  DayOfWeek        Date  Sales  Customers  Open  Promo StateHoliday  \
0      1          5  2015-07-31   5263        555     1      1            0   
1      2          5  2015-07-31   6064        625     1      1            0   
2      3          5  2015-07-31   8314        821     1      1            0   
3      4          5  2015-07-31  13995       1498     1      1            0   
4      5          5  2015-07-31   4822        559     1      1            0   

   SchoolHoliday  
0              1  
1              1  
2              1  
3              1  
4              1  

Lojas:
   Store StoreType Assortment  CompetitionDistance  CompetitionOpenSinceMonth  \
0      1         c          a               1270.0                        9.0   
1      2         a          a                570.0                       11.0   
2      3         a          a              14130.0                       12.0   
3      4         c          c                620.0                     

/tmp/ipython-input-2724456449.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('/content/train.csv')


In [9]:
df = pd.merge(train, store, on='Store', how='left')

df = df[df['Open'] == 1]

df = df[df['Sales'] > 0]

df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [10]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['WeekOfYear'] = df['Date'].dt.isocalendar().week

df['StateHoliday'] = df['StateHoliday'].astype(str)
df = pd.get_dummies(df, columns=['StoreType', 'Assortment', 'StateHoliday'], drop_first=True)

features = [
    'Store', 'DayOfWeek', 'Promo', 'SchoolHoliday', 'CompetitionDistance',
    'Promo2', 'Year', 'Month', 'WeekOfYear'
] + [col for col in df.columns if 'StoreType_' in col or 'Assortment_' in col or 'StateHoliday_' in col]

X = df[features]
y = df['Sales']

X = X.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Shape treino:", X_train.shape)
print("Shape teste:", X_test.shape)

Shape treino: (675470, 17)
Shape teste: (168868, 17)


In [11]:
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

In [12]:
tree = DecisionTreeRegressor(max_depth=8, random_state=42)
tree.fit(X_train, y_train)
pred_tree = tree.predict(X_test)

In [13]:
xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)

In [17]:
def avaliar_modelo(y_true, y_pred, nome):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"\n{nome}")
    print(f"RMSE = {rmse:.2f}")
    print(f"MAE  = {mae:.2f}")
    print(f"R²   = {r2:.3f}")
    return [nome, rmse, mae, r2]

resultados = []
resultados.append(avaliar_modelo(y_test, pred_lr, "Regressão Linear"))
resultados.append(avaliar_modelo(y_test, pred_tree, "Árvore de Decisão"))
resultados.append(avaliar_modelo(y_test, pred_xgb, "XGBoost"))

df_resultados = pd.DataFrame(resultados, columns=["Modelo", "RMSE", "MAE", "R²"])
df_resultados



Regressão Linear
RMSE = 2761.25
MAE  = 2018.84
R²   = 0.210

Árvore de Decisão
RMSE = 2463.93
MAE  = 1808.68
R²   = 0.371

XGBoost
RMSE = 1078.50
MAE  = 759.51
R²   = 0.879


,Modelo,RMSE,MAE,R²
0,Regressão Linear,2761.251789,2018.837322,0.209860
1,Árvore de Decisão,2463.933033,1808.679422,0.370856
2,XGBoost,1078.498319,759.509888,0.879460


In [19]:
importances = pd.DataFrame({
    'Variável': X.columns,
    'Importância': xgb.feature_importances_
}).sort_values(by='Importância', ascending=False)

importances.head(10)

,Variável,Importância
9,StoreType_b,0.275967
2,Promo,0.152047
4,CompetitionDistance,0.090865
13,Assortment_c,0.088088
5,Promo2,0.085147
0,Store,0.071386
10,StoreType_c,0.061844
11,StoreType_d,0.057538
1,DayOfWeek,0.026484
7,Month,0.021379
